In [ ]:
!pip install scipy tqdm -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict
from scipy.stats import ttest_ind
from tqdm import tqdm

import random

In [ ]:
WORLD_X = 26
WORLD_Y = 14

GOAL_X = 23.0
GOAL_Y = 4.5
GOAL_R = 1.0

CHARGE_POWER = 0.030

START_MEAN = np.array([4.0,10.0])
START_SIGMA = 2.0 # Стандартное отклонение начальной позиции (вариативность для старта)

START_X_CLIP = (1.5,8.0) # Пределы начальных координат (чтобы не выходить за границы и не рождаться в препятствиях)
START_Y_CLIP = (4.0,13.0)

VELS = [0.12,0.28,0.45,0.65] # скорости

OBSTACLES = [ # координаты препятствий
    (9.0,3.0,10.5,8.0),
    (12.0,10.0,14.0,12.5),
    (15.5,2.0,17.5,6.0),
    (18.0,7.5,20.0,11.0),
    (21.0,1.0,22.5,3.5)
]

MAX_STEPS = 210 # максимальная длина эпизода

SUCCESS_BATTERY = 0.88 # уровень заряда (успех)
FAIL_BATTERY = 0.03 # порог заряда (ниже - провал)

Среда

Пояснение изменений: в ходе 3200 (10*10*8*4) состояний. В методичке 9000.


*   Уменьшение числа состояний ускоряет обучение, так как таблица меньше и каждое состояние посещается чаще;
*   10 градаций по x и y на поле 26*14 дают разрешение около 2.6 единицы, этого достаточно, чтобы различать препятствия (минимальный размер препятствия 1.5*3)
*   8 градаций заряда (шаг 0.125) позволяют агенту чувствовать разницу между критическими уровнями (0.03, 0.88) и промежуточными;
*   Точность ниже, но сходимость - выше.

Использование максимальной скорости: при этой скорости высокий разряд батареи, поэтому агент сам быстро научится не использовать ее лишний раз, в то время как это дает агенту больше гибкости (находить быструю траекторию)

Ветер анизотропный: по горизонтали (x) сдвиг от -0.14 до +0.05 (чаще отрицательный, т.е. тянет влево), по вертикали (y) от -0.07 до +0.12 (тянет скорее вверх). Множители 0.8 и 1.2 позже в step() усилят влияние на движение.

In [ ]:
class DroneEnv:

    def __init__(self):
        # Интервалы, на которые разбивают координаты до цели (x,y) и заряд батареи
        self.dx_bins = 10
        self.dy_bins = 10
        self.b_bins = 8

        self.wind_period = 25 # Изменение ветра каждые 25 секунд

        self.actions = []

        dirs = [ # направления: вправо, вверх, влево, вниз
            0,
            np.pi/2,
            np.pi,
            3*np.pi/2
        ]

        speeds = [
            0.12,
            0.28,
            0.45,
            0.65
        ]

        self.actions = []

        for d in dirs:
            for s in speeds:
                self.actions.append((d,s))

        self.n_actions = len(self.actions) # 4 * 4 = 16 действий


    def sample_start(self):
        # (4.0, 10.0) и сигма 2.0
        p = np.random.normal(
            START_MEAN,
            START_SIGMA
        )
        # x в [1.5, 8.0]
        p[0] = np.clip(
            p[0],
            *START_X_CLIP
        )
        # y в [4.0, 13.0]
        p[1] = np.clip(
            p[1],
            *START_Y_CLIP
        )

        return p

    def reset(self):

        self.x,self.y = self.sample_start() # начальные координаты

        self.battery = 1.0

        self.speed = 0.28

        self.step_count = 0

        self.collisions = 0 # счётчик столкновений за эпизод

        self.wind = self.sample_wind() # первый случайный ветер

        return self.get_state() # вернуть дискретное состояние

    def sample_wind(self):

        wx = np.random.uniform(-0.14,0.05)
        wy = np.random.uniform(-0.07,0.12)

        return np.array([wx,wy])

    def in_goal(self): # расстояние до центра станции меньше радиуса GOAL_R = 1.0

        d = np.sqrt(
            (self.x-GOAL_X)**2+
            (self.y-GOAL_Y)**2
        )

        return d < GOAL_R

    def point_in_obstacle(self,x,y):

        for xmin,ymin,xmax,ymax in OBSTACLES:

            if xmin <= x <= xmax and ymin <= y <= ymax:
                return True

        return False

    def get_state(self):

        dx = GOAL_X - self.x # смещение до цели по x
        dy = GOAL_Y - self.y # смещение по y

        dx_bin = np.digitize( # разбивает эти значения на интервалы
            dx,
            np.linspace( # создаёт границы интервалов от -26 до 26 с шагом ~5.78
                -WORLD_X,
                WORLD_X,
                self.dx_bins - 1
            )
        )

        dy_bin = np.digitize(
            dy,
            np.linspace(
                -WORLD_Y,
                WORLD_Y,
                self.dy_bins - 1
            )
        )

        b_bin = np.digitize(
            self.battery,
            np.linspace(
                0,
                1,
                self.b_bins - 1
            )
        )

        v_bin = np.argmin(
            np.abs(
                np.array(VELS) - self.speed
            )
        )
        # кортеж из четырёх целых чисел: (dx_bin, dy_bin, b_bin, v_bin). Агент будет использовать его для работы с Q-таблицей
        return (
            dx_bin,
            dy_bin,
            b_bin,
            v_bin
        )

    def step(self,action_id): # вызывается каждый раз, когда агент выбирает действие. Он обновляет среду и возвращает новое состояние, награду, флаг done и информацию

        self.step_count += 1

        if self.step_count % self.wind_period == 0:
            self.wind = self.sample_wind() # Ветер остаётся постоянным в течение 25 шагов, затем меняется

        theta,v = self.actions[action_id] # Из списка всех 16 действий берётся угол и скорость

        old_dist = np.sqrt(
            (self.x - GOAL_X)**2 +
            (self.y - GOAL_Y)**2
        )

        self.speed = v
        # Расчёт следующей позиции (с ветром)
        nx = self.x + v*np.cos(theta) + 0.8*self.wind[0]
        ny = self.y + v*np.sin(theta) + 1.2*self.wind[1]

        nx = np.clip(nx,0,WORLD_X)
        ny = np.clip(ny,0,WORLD_Y)
        # Если новая позиция попадает в препятствие, движение отменяется — дрон остаётся на месте. Счётчик столкновений увеличивается
        collision = False

        if self.point_in_obstacle(nx,ny):
            collision = True
            self.collisions += 1
            nx,ny = self.x,self.y

        self.x,self.y = nx,ny

        new_dist = np.sqrt(
            (self.x - GOAL_X)**2 +
            (self.y - GOAL_Y)**2
        )
        # Эта величина положительна, если дрон приблизился к цели. Будет добавлена к награде
        progress_reward = (
            old_dist - new_dist
        )
        # Обновление батареи (зависимость от квадрата скорости, дополнительная трата от ветра, если внутри станции - заряд, меньше 1)
        station = self.in_goal()

        self.battery = min(
            1.0,
            self.battery
            -0.020*v*v
            -0.004*abs(self.wind[0])
            -0.003*abs(self.wind[1])
            +(CHARGE_POWER if station else 0)
        )
        # Проверка окончания эпизода и награда
        done = False

        success = False

        if station and self.battery > SUCCESS_BATTERY and self.step_count < 150:

            reward = 100
            done = True
            success = True

        elif self.battery < FAIL_BATTERY:

            reward = -180
            done = True

        elif self.step_count > MAX_STEPS:

            reward = -180
            done = True

        else:

            reward = (
                -1.2 # Базовый штраф
                -0.035*v*v # Штраф за скорость
                + 2.0 * progress_reward # Бонус за прогресс
                + 0.15 * CHARGE_POWER * (1 if station else 0) # Маленький бонус за нахождение на станции (чтобы остался на зарядке)
            )

        info = {
            "success":success,
            "collision":collision
        }

        return self.get_state(),reward,done,info
        # Возвращаются: новое состояние (дискретные индексы), награда, флаг завершения, служебная информация (успех или столкновение)

Базовый агент

Базовый табличный Q-learning агент (рациональный, risk-neutral). Хранит Q(s,a) в словаре с дефолтным значением (ноль для всех действий).

In [ ]:
class QAgent:

    def __init__(
        self,
        n_actions, # # количество возможных действий (16)
        alpha=0.05, # # скорость обучения (learning rate) – насколько быстро обновляются Q-значени
        gamma=0.99 # # коэффициент дисконтирования – важность будущих наград
    ):

        self.alpha = alpha
        self.gamma = gamma

        self.n_actions = n_actions
        # Q-таблица: словарь, где ключ = состояние (кортеж из 4 индексов),
        # а значение = numpy-вектор длины n_actions (ценность каждого действия в этом состоянии)
        # defaultdict(lambda: np.zeros(n_actions)): если состояние встречается впервые,
        # автоматически создаётся вектор из нулей.
        self.Q = defaultdict(
            lambda: np.zeros(n_actions)
        )

    def act(self,state,eps):
        """
        Выбор действия по epsilon-жадной стратегии (ε-greedy).

        Параметры:
            state - текущее состояние (кортеж (dx_bin, dy_bin, b_bin, v_bin))
            eps   - вероятность случайного действия (exploration rate)

        Возвращает:
            выбранное действие (целое число от 0 до n_actions-1)
        """
        if np.random.rand() < eps:
            return np.random.randint(self.n_actions)

        return np.argmax(
            self.Q[state]
        )

Rational Q-learning

 Рациональный (risk‑neutral) агент, реализующий классический Q-learning.
    Наследует все поля и метод act() от QAgent.
    Добавляет метод update(), который корректирует Q-значения по наблюдаемому переходу.
     Выполняет одно обновление Q-таблицы после выполнения действия a в состоянии s,
        получения награды r и перехода в новое состояние s2.

        Параметры:
            s  – предыдущее состояние (кортеж из 4 индексов)
            a  – выполненное действие (int от 0 до n_actions-1)
            r  – полученная награда (float)
            s2 – следующее состояние (кортеж из 4 индексов)

In [ ]:
class RationalAgent(QAgent):

    def update(self,s,a,r,s2):

        td = r + self.gamma*np.max(self.Q[s2]) # # Вычисляем TD‑цель (target): r + γ * max_{a'} Q(s2, a'). Максимальная ценность действия в новом состоянии
        #  # Обновляем Q(s, a) по формуле: Q <- Q + α * (TD_target - Q)
        self.Q[s][a] += self.alpha*(
            td-self.Q[s][a]
        )

Prospect Theory

Поведенческий агент, использующий трансформацию награды по теории перспектив (Prospect Theory).
    Вместо реальной награды r подставляется субъективная ценность v(r),
    что приводит к несимметричному восприятию выигрышей и потерь.
    """
        Обновление Q-таблицы с искажённой наградой.
        Вместо реального r используется v(r).

        Параметры:
            s  – предыдущее состояние
            a  – выполненное действие
            r  – реальная награда, полученная от среды
            s2 – следующее состояние

        Функция субъективной ценности (value function) из теории перспектив.
        Формула (12) в методичке.

        Параметры:
            r – реальная награда (положительная или отрицательная)

        Возвращает:
            искажённую ценность v(r), которая используется вместо r в обучении.

In [ ]:
class ProspectAgent(QAgent):

    def __init__(
        self,
        n_actions,
        alpha=0.1, # скорость обучения (learning rate)
        gamma=0.99, # коэффициент дисконтирования
        alpha_p=0.88, # # чувствительность к выигрышам (меньше 1)
        beta_p=0.88, # чувствительность к потерям (меньше 1)
        lambda_p=2.35 # коэффициент неприятия потерь
    ):

        super().__init__(
            n_actions,
            alpha,
            gamma
        )

        self.alpha_p = alpha_p
        self.beta_p = beta_p
        self.lambda_p = lambda_p
    # Функция субъективной ценности (value function) из теории перспектив
    def v(self,r):

        if r >= 0: # Для выигрышей: r^{α_p}. α_p < 1 сжимает крупные выигрыши
            return r**self.alpha_p
        # Для потерь: -λ_p * (-r)^{β_p}. β_p < 1 сжимает крупные потери,
        # а λ_p > 1 делает потери более тяжёлыми, чем равные по модулю выигрыши
        return -self.lambda_p*((-r)**self.beta_p)

    def update(self,s,a,r,s2):

        r_pt = self.v(r) # трансформировать реальную награду в искаженную

        td = r_pt + self.gamma*np.max(self.Q[s2]) # Вычисление цели на основе искаженной награды
        # Обновление Q(s, a) по правилу стохастического приближения
        # self.Q[s][a] += α * (td - текущее_значение)
        self.Q[s][a] += self.alpha*(
            td-self.Q[s][a]
        )

Risk Sensitive

    Риск-чувствительный агент (энтропийный) согласно формулам (18)-(21).

In [ ]:
class RiskSensitiveAgent(QAgent):
    def __init__(self, n_actions, eta=0.1, alpha=0.05, gamma=0.99):
        super().__init__(n_actions, alpha, gamma)
        self.eta = eta

    def update(self, s, a, r, s2):
        # sign(r) * (exp(eta*|r|) - 1) / eta   – экспоненциальное искажение
        if r >= 0:
            r_t = (np.exp(self.eta * r) - 1.0) / self.eta
        else:
            r_t = - (np.exp(-self.eta * r) - 1.0) / self.eta   # r отрицательное, -r положительно

        # Стандартное Q‑обновление, но с искажённой наградой
        td = r_t + self.gamma * np.max(self.Q[s2])
        self.Q[s][a] += self.alpha * (td - self.Q[s][a])

Обучение

In [ ]:
def train_agent(
    env,
    agent,
    episodes=50000
):

    rewards = []

    eps = 1.0

    for ep in tqdm(range(episodes)):

        s = env.reset()

        total_reward = 0

        done = False

        while not done:
            # Агент выбирает действие по epsilon-жадной политике
            a = agent.act(s,eps)
            # Среда делает шаг: новое состояние, награда, флаг окончания, доп. информация
            s2,r,done,_ = env.step(a)
            # Агент обновляет свои Q-значения на основе перехода
            agent.update(
                s,a,r,s2
            )

            total_reward += r # накапливаем суммарную награду эпизода

            s = s2 # переходим в новое состояние

        rewards.append(total_reward)
        # Уменьшаем epsilon (ослабляем исследование, усиливаем эксплуатацию)
        # eps = max(ε_min, eps * decay)
        eps = max(
            0.02,
            eps*0.9995
        )

    return rewards

Обучение агентов

In [ ]:
env = DroneEnv()

rational = RationalAgent(env.n_actions)
prospect = ProspectAgent(env.n_actions)
risk = RiskSensitiveAgent(
    env.n_actions,
    eta=0.1
)

r_rewards = train_agent(env,rational)
p_rewards = train_agent(env,prospect)
rs_rewards = train_agent(env,risk)

Learning Curves

In [ ]:
def moving_avg(x,w=150):

    return np.convolve(
        x,
        np.ones(w)/w,
        mode='valid'
    )

plt.figure(figsize=(12,6))

plt.plot(
    moving_avg(r_rewards),
    label='Rational'
)

plt.plot(
    moving_avg(p_rewards),
    label='Prospect'
)

plt.plot(
    moving_avg(rs_rewards),
    label='Risk'
)

plt.legend()
plt.grid()
plt.show()

Тестирование

In [ ]:
def test_agent(
    env,
    agent,
    episodes=300
):

    success = 0

    times = []

    energies = []

    final_batteries = []

    collision_eps = 0

    trajectories = []

    for ep in range(episodes):

        s = env.reset()

        done = False

        traj = []

        start_b = env.battery

        while not done:

            a = agent.act(
                s,
                0.005
            )

            traj.append(
                (env.x,env.y)
            )

            s,r,done,info = env.step(a)

        trajectories.append(traj)

        final_batteries.append(
            env.battery
        )

        energies.append(
            start_b-env.battery
        )

        if env.collisions > 0:
            collision_eps += 1

        if info["success"]:

            success += 1

            times.append(
                env.step_count
            )

    return {
        "success_rate":
        100*success/episodes,

        "times":times,

        "energy":energies,

        "battery":final_batteries,

        "collision_rate":
        100*collision_eps/episodes,

        "traj":trajectories
    }

In [ ]:
res_r = test_agent(env,rational)
res_p = test_agent(env,prospect)
res_rs = test_agent(env,risk)

In [ ]:
for name,res in [
    ("Rational",res_r),
    ("Prospect",res_p),
    ("Risk",res_rs)
]:

    print(name)

    print(
        "Success:",
        res["success_rate"]
    )

    print(
        "Mean Time:",
        np.mean(res["times"])
        if len(res["times"]) else np.nan
    )

    print(
        "Energy:",
        np.mean(res["energy"])
    )

    print(
        "Collisions:",
        res["collision_rate"]
    )

    print()



1.   Среда агрессивна к потерям: штраф за каждый шаг (−1.2), за скорость (−0.035v²), плюс редкий, но огромный штраф за разряд (−180).
2.   Prospect (α=0.88, β=0.88, λ=2.35) гипертрофирует страх потерь: даже обычный шаг кажется очень болезненным, а успех (+100) обесценивается до +63. В итоге агент избегает любых движений → хаотичное поведение, 98% коллизий.
3. Энтропийный агент с η=1.0 заплатил за избегание риска: он выбирал более безопасные, но слишком длинные маршруты, из-за чего часто не успевал к станции за 210 шагов. При этом в успешных эпизодах он всё же показывал время, сравнимое с рациональным агентом.

Классический risk‑нейтральный Q-learning оптимален для данной награды. Поведенческие искажения вредны, когда отрицательные сигналы поступают на каждом шагу. Чтобы они работали, нужно снижать λ и увеличивать η, но по заданию параметры фиксированы.

In [ ]:
plt.figure(figsize=(10,6))

for label,res in [
    ("Rational",res_r),
    ("Prospect",res_p),
    ("Risk",res_rs)
]:

    t = np.sort(res["times"])

    if len(t)==0:
        continue

    y = np.arange(len(t))/len(t)

    plt.plot(t,y,label=label)

plt.legend()
plt.grid()
plt.show()

HEATMAP

In [ ]:
from matplotlib.patches import Rectangle, Circle

def build_battery_heatmap(agent, nx=40, ny=25):

    xs = np.linspace(1, WORLD_X - 1, nx)
    ys = np.linspace(1, WORLD_Y - 1, ny)

    grid = np.full((ny, nx), np.nan)

    for iy, y in enumerate(ys):
        for ix, x in enumerate(xs):

            env = DroneEnv()

            # точки внутри препятствий не рассматриваем
            if env.point_in_obstacle(x, y):
                continue

            # задаём стартовое состояние
            env.x = x
            env.y = y
            env.battery = 1.0
            env.speed = 0.28
            env.step_count = 0
            env.collisions = 0
            env.wind = env.sample_wind()

            state = env.get_state()
            done = False

            while not done:
                action = agent.act(state, eps=0.0)
                state, reward, done, info = env.step(action)

            # сохраняем остаток батареи
            grid[iy, ix] = env.battery

    return grid, xs, ys

In [ ]:
from matplotlib.patches import Rectangle, Circle
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

agents = [
    ("Rational", rational),
    ("Prospect", prospect),
    ("Risk-sensitive", risk)
]

# Общая нормировка для всех тепловых карт
norm = Normalize(vmin=0, vmax=1)
cmap = "RdYlGn"

for ax, (name, agent) in zip(axes, agents):
    grid, xs, ys = build_battery_heatmap(agent)

    im = ax.imshow(
        grid,
        origin="lower",
        extent=[xs[0], xs[-1], ys[0], ys[-1]],
        cmap=cmap,
        interpolation="nearest",
        aspect="equal",
        norm=norm
    )

    # Контуры препятствий
    for (x1, y1, x2, y2) in OBSTACLES:
        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor="black",
                linewidth=2
            )
        )

    # Станция зарядки
    ax.add_patch(
        Circle(
            (GOAL_X, GOAL_Y),
            GOAL_R,
            fill=False,
            edgecolor="blue",
            linewidth=2
        )
    )

    ax.set_xlim(0, WORLD_X)
    ax.set_ylim(0, WORLD_Y)
    ax.set_title(name)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

# Освобождаем место справа и создаём отдельную ось для colorbar
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])   # [left, bottom, width, height]

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # фиктивный массив, чтобы colorbar сработал
fig.colorbar(sm, cax=cbar_ax, label="Final battery")

plt.show()

In [ ]:
def draw_env():

    for o in OBSTACLES:

        x1,y1,x2,y2 = o

        plt.fill(
            [x1,x2,x2,x1],
            [y1,y1,y2,y2],
            alpha=0.4
        )

    circle = plt.Circle(
        (GOAL_X,GOAL_Y),
        GOAL_R,
        fill=False,
        linewidth=3
    )

    plt.gca().add_patch(circle)


def plot_traj(trajs,title):

    plt.figure(figsize=(8,5))

    draw_env()

    for t in trajs[:4]:

        arr = np.array(t)

        plt.plot(
            arr[:,0],
            arr[:,1]
        )

    plt.xlim(0,WORLD_X)
    plt.ylim(0,WORLD_Y)

    plt.title(title)

    plt.show()

In [ ]:
plot_traj(
    res_r["traj"],
    "Rational"
)

plot_traj(
    res_p["traj"],
    "Prospect"
)

plot_traj(
    res_rs["traj"],
    "Risk Sensitive"
)